# Creación del inventario de Oxford-IIIT Pet

In [ ]:
# Importar bibiotecas necesarias
from pathlib import Path				# Para manejar rutas de archivos y directorios
import numpy as np						# Para procesar mascaras con etiquetas numéricas
from PIL import Image					# Para procesar imágenes
import json								# Para procesar archivos JSON
import xml.etree.ElementTree as ET 		# Para procesar archivos XML
import pandas as pd						# Para procesar archivos CSV
from matplotlib import pyplot as plt	# Para visualizar imágenes y máscaras


In [ ]:
# Definir rutas
PROYECTO = Path("..")	# Directorio raíz del proyecto
DATA = PROYECTO / "data" / "raw" / "oxford-iiit-pet"	# Directorio de datos
IMAGES = DATA / "images"	# Directorio de imágenes
MASKS = DATA / "annotations" / "trimaps"	# Directorio de máscaras
BBOXES = DATA / "annotations" / "xmls"	# Directorio de cajas delimitadoras
LIST = DATA / "annotations" / "list.txt"	# Archivo de lista de imágenes y etiquetas


In [ ]:
# Revisar si los directorios existen
if not DATA.exists():
	raise FileNotFoundError(f"El directorio de datos no existe: {DATA}")
elif not IMAGES.exists():
	raise FileNotFoundError(f"El directorio de imágenes no existe: {IMAGES}")
elif not MASKS.exists():
	raise FileNotFoundError(f"El directorio de máscaras no existe: {MASKS}")
elif not BBOXES.exists():
	raise FileNotFoundError(f"El directorio de cajas delimitadoras no existe: {BBOXES}")

## Funciones para construir el manifest

### Función para procesar list.txt

In [ ]:
def cargar_clases(ruta_lista):
	# Cargar las clases desde un archivo de texto
	clases = {}

	with ruta_lista.open("r", encoding="utf-8") as archivo:

		for linea in archivo:

			linea = linea.strip()

			# Ignorar comentarios y líneas vacías
			if not linea or linea.startswith("#"):
				continue

			# Separar los cuatro campos
			image_id, class_id, species_id, breed_id = linea.split()

			# Obtener la raza eliminando el último _número
			raza = image_id.rsplit("_", 1)[0]

			# Guardar los identificadores de la raza
			clases[raza] = {
				"class_id": int(class_id),
				"species_id": int(species_id),
				"breed_id": int(breed_id)
			}

	return clases

### Funciones para procesar atributos de las imágenes

In [ ]:
def obtener_clase(image_id, clases):
	# Obtener la información de la clase a partir del identificador de imagen

	raza = image_id.rsplit("_", 1)[0] # Extraer el nombre de la raza

	if raza not in clases:
		raise ValueError(f"Raza no encontrada: {raza}")

	datos = clases[raza]

	# Convertir identificador de especie a nombre
	especies = {
		1: "cat",
		2: "dog"
	}

	return {
		"breed": raza,
		"class_id": datos["class_id"],
		"species_id": datos["species_id"],
		"species": especies[datos["species_id"]],
		"breed_id": datos["breed_id"]
	}

In [ ]:
def verificar_dimensiones(ruta_imagen, ruta_mascara):
	# Verificar que las dimensiones de la imagen y la máscara sean iguales
	# y que los archivos se puedan leer correctamente

	# Verificar existencia de archivos
	if not ruta_imagen.is_file():
		return 1, None, None # Imagen inexistente

	if not ruta_mascara.is_file():
		return 2, None, None # Máscara inexistente

	try:
		# Obtener dimensiones de la imagen
		with Image.open(ruta_imagen) as imagen:
			ancho, alto = imagen.size

		# Obtener dimensiones de la máscara
		with Image.open(ruta_mascara) as mascara:
			ancho_mascara, alto_mascara = mascara.size
	
	except (OSError, ValueError):
		return 4, None, None # Archivo existente, pero ilegible o inválido

	# Comparar dimensiones
	if (ancho, alto) != (ancho_mascara, alto_mascara):
		return 3, ancho, alto # Dimensiones diferentes

	# Validación exitosa
	return 0, ancho, alto

In [ ]:
def leer_bounding_box(ruta_xml, ancho, alto):
	# Verifica que el bounding box esté dentro de los límites de la imagen
	# y que el archivo de configuracion xml sea legible.

	# Verificar existencia del XML
	if not ruta_xml.is_file():
		return 1, None # Archivo XML no encontrado

	try:
		raiz = ET.parse(ruta_xml).getroot()

		# Obtener dimensiones declaradas en el XML
		ancho_xml = int(raiz.findtext("size/width"))
		alto_xml = int(raiz.findtext("size/height"))

		# Verificar consistencia con las dimensiones reales
		if (ancho_xml, alto_xml) != (ancho, alto):
			return 5, None # Dimensiones en XML no coinciden con las reales

		# Extraer el bounding box
		bbox = raiz.find("object/bndbox")

		if bbox is None:
			return 2, None # Bounding box no encontrado

		coordenadas = {
			"xmin": int(bbox.findtext("xmin")),
			"ymin": int(bbox.findtext("ymin")),
			"xmax": int(bbox.findtext("xmax")),
			"ymax": int(bbox.findtext("ymax"))
		}

	except (ET.ParseError, OSError, ValueError, TypeError):
		return 4, None # Archivo XML ilegible o inválido

	# Extraer coordenadas
	xmin = coordenadas["xmin"]
	ymin = coordenadas["ymin"]
	xmax = coordenadas["xmax"]
	ymax = coordenadas["ymax"]

	# Validar límites del bounding box
	if not (0 <= xmin < xmax <= ancho):
		return 3, None # Coordenadas del bounding box fuera de los límites de la imagen

	if not (0 <= ymin < ymax <= alto):
		return 3, None # Coordenadas del bounding box fuera de los límites de la imagen

	return 0, coordenadas # Validación exitosa, devolver coordenadas

In [ ]:
def procesar_imagen(ruta_imagen, directorio_mascaras, directorio_xmls, directorio_data, clases):

	# Obtener el identificador de la imagen
	image_id = ruta_imagen.stem

	# Construir las rutas correspondientes
	ruta_mascara = directorio_mascaras / f"{image_id}.png"
	ruta_xml = directorio_xmls / f"{image_id}.xml"

	# Obtener metadatos de la clase
	datos_clase = obtener_clase(image_id, clases)

	# Verificar dimensiones de imagen y máscara
	codigo_dim, ancho, alto = verificar_dimensiones(ruta_imagen, ruta_mascara)

	if codigo_dim != 0:
		return f"DIM_{codigo_dim}", None

	# Leer y validar XML y bounding box
	codigo_bbox, bbox = leer_bounding_box(ruta_xml, ancho, alto)

	if codigo_bbox != 0:
		print(f"Advertencia: {image_id} → XML_{codigo_bbox}")

		bbox = {
			"xmin": None,
			"ymin": None,
			"xmax": None,
			"ymax": None
		}

	# Construir registro del manifest
	registro = {
		"image_id": image_id,
		**datos_clase,
		"image_path": ruta_imagen.relative_to(directorio_data).as_posix(),
		"mask_path": ruta_mascara.relative_to(directorio_data).as_posix(),
		"xml_path": ruta_xml.relative_to(directorio_data).as_posix(),
		"width": ancho,
		"height": alto,
		**bbox
	}

	return 0, registro

### Función para generar el manifest

In [ ]:
def generar_manifest(directorio_data,directorio_imagenes, directorio_mascaras, directorio_xmls, ruta_lista, ruta_salida):

	# Cargar catálogo de clases una sola vez
	clases = cargar_clases(ruta_lista)

	registros = []
	errores = []

	# Definir el directorio de imágenes
	directorio_imagenes = directorio_data / "images"

	# Buscar recursivamente archivos JPG
	imagenes = sorted(directorio_imagenes.rglob("*.jpg"))

	for ruta_imagen in imagenes:
		 
		try:
			codigo, registro = procesar_imagen(ruta_imagen, directorio_mascaras, directorio_xmls, directorio_data, clases)

			if codigo == 0:
				registros.append(registro)
				print(f"OK: {ruta_imagen.name}")

			else:
				errores.append({
					"image_id": ruta_imagen.stem,
					"error": codigo
				})
				print(f"ERROR: {ruta_imagen.name} → {codigo}")

		except ValueError as error:
			errores.append({
				"image_id": ruta_imagen.stem,
				"error": str(error)
			})
			print(f"ERROR: {ruta_imagen.name} → {error}")

	# Crear DataFrame
	manifest = pd.DataFrame(registros)

	# Guardar CSV
	ruta_salida.parent.mkdir(parents=True, exist_ok=True)
	manifest.to_csv(ruta_salida, index=False)

	print("\nResumen")
	print(f"Imágenes encontradas: {len(imagenes)}")
	print(f"Registros válidos: {len(registros)}")
	print(f"Registros con errores: {len(errores)}")
	print(f"Manifest guardado en: {ruta_salida}")

	return manifest, pd.DataFrame(errores)

In [ ]:
# Integración del manifest
ruta_lista = DATA / "annotations" / "list.txt"
ruta_manifest = PROYECTO / "data" / "manifest" / "manifest.csv"
manifest, errores = generar_manifest(DATA, IMAGES, MASKS, BBOXES, ruta_lista, ruta_manifest)

#Visualización de los resultados
print("\nManifest:")
display(manifest)
print("\nErrores:")
display(errores)

# Archivo de configuración del dataset

In [ ]:
# Definir la configuración
config = {
    "dataset": {
        "name": "oxford-iiit-pet",
        "root": "data/raw/oxford-iiit-pet",
        "images": "images",
        "masks": "annotations/trimaps",
        "xmls": "annotations/xmls",
        "labels": "annotations/list.txt"
    },
    "manifest": {
        "path": "data/manifest/manifest.csv"
    }
}

In [ ]:
# Definir la ruta del archivo
ruta_config = PROYECTO / "config" / "dataset.json"

# Crear el directorio si no existe
ruta_config.parent.mkdir(parents=True, exist_ok=True)

# Guardar el archivo JSON
with ruta_config.open("w", encoding="utf-8") as archivo:
    json.dump(config, archivo, indent=4, ensure_ascii=False)


# Probar el archivo de configuración y el manifest

In [ ]:
# Cargar la configuración desde el archivo JSON
with ruta_config.open("r", encoding="utf-8") as archivo:
    config = json.load(archivo)

# Definir rutas de directorios y archivos según la configuración
DIRECTORIO_DATA = PROYECTO / config["dataset"]["root"]
IMAGES = DIRECTORIO_DATA / config["dataset"]["images"]
MASKS = DIRECTORIO_DATA / config["dataset"]["masks"]
BBOXES = DIRECTORIO_DATA / config["dataset"]["xmls"]
RUTA_LISTA = DIRECTORIO_DATA / config["dataset"]["labels"]
RUTA_MANIFEST = PROYECTO / config["manifest"]["path"]

# Visualización de las rutas
print(f"Directorio data: {DIRECTORIO_DATA}")
print(f"Ruta de imágenes: {IMAGES}")
print(f"Ruta de máscaras: {MASKS}")
print(f"Ruta de bounding boxes: {BBOXES}")
print(f"Ruta de lista: {RUTA_LISTA}")
print(f"Ruta de manifest: {RUTA_MANIFEST}")

In [ ]:
# Visualizar una imagen y su máscara correspondiente desde el manifest
# Cargar manifest
manifest = pd.read_csv(RUTA_MANIFEST)

print(f"Registros disponibles: {len(manifest)}")
display(manifest.head())

In [ ]:
# Seleccionar una muestra aleatoria de registros del manifest
n_muestras = 6
muestra = manifest.sample(n=6, random_state=42)
display(muestra[["image_id", "breed", "species", "width", "height"]])

In [ ]:
def visualizar_muestra(muestra, dataset_root):

    n = len(muestra)

    fig, axes = plt.subplots(
        n, 2,
        figsize=(9, 3.5 * n),
        squeeze=False
    )

    for i, (_, fila) in enumerate(muestra.iterrows()):

        ruta_imagen = dataset_root / fila["image_path"]
        ruta_mascara = dataset_root / fila["mask_path"]

        with Image.open(ruta_imagen) as img:
            imagen = np.array(img.convert("RGB"))

        with Image.open(ruta_mascara) as mask:
            mascara = np.array(mask)

        # Imagen original
        axes[i, 0].imshow(imagen)
        axes[i, 0].set_title(f"{fila['image_id']} - Imagen")

        # Máscara de segmentación
        axes[i, 1].imshow(
            mascara,
            cmap="gray",
            vmin=1,
            vmax=3,
            interpolation="nearest"
        )
        axes[i, 1].set_title("Máscara (trimap)")

        for ax in axes[i]:
            ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# Visualizar la muestra seleccionada
visualizar_muestra(muestra, DIRECTORIO_DATA)